# pywhyllm Walkthrough

This notebook walks through the core features of `pywhyllm` — an LLM-backed library for causal discovery.

**What you'll learn:**
- Building a causal graph over a set of variables
- Querying the graph locally (parents, children, ancestors — no extra LLM calls)
- Using expert consensus to get more reliable graphs
- Finding confounders (observed and latent)
- Switching to a different LLM provider

In [ ]:
%pip install -e ../.. instructor openai python-dotenv --quiet

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # reads OPENAI_API_KEY from a .env file

# Or set directly
import os; os.environ["OPENAI_API_KEY"] = ""

In [ ]:
import instructor
from openai import AsyncOpenAI

from pywhyllm.suggesters import ModelSuggester, CausalGraph

# Wrap your async client with instructor for structured outputs.
# Swap AsyncOpenAI() for AsyncAnthropic(), etc. to change providers — 
# nothing else in this notebook changes.
client = instructor.from_openai(AsyncOpenAI())

suggester = ModelSuggester(
    client=client,
    model="gpt-5.4-mini-2026-03-17",
    context="general causal relationships",  # optional — injected into system prompts
)

## Building a causal graph

`suggest_graph` is the core method. It makes **one LLM call** and returns a `CausalGraph` object containing all identified causal edges.

In [ ]:
variables = ["ice cream sales", "temperature", "shark attacks", "cavities"]

# One LLM call — returns a CausalGraph
graph = await suggester.suggest_graph(variables)

In [ ]:
print(graph)

## Querying the graph locally

Once the graph is built, structural queries — parents, children, ancestors, descendants — are **local operations**. No further LLM calls are made.

In [ ]:
# Direct causes and effects — no LLM call
print("Parents of 'shark attacks':", graph.parents_of("shark attacks"))
print("Children of 'temperature':", graph.children_of("temperature"))

## Ancestors and descendants

Go beyond direct causes — traverse the full upstream or downstream chain. Still zero LLM calls.

In [ ]:
print("Ancestors of 'cavities':", graph.ancestors_of("cavities"))
print("Descendants of 'temperature':", graph.descendants_of("temperature"))

## Mediators and instrumental variables

A **mediator** sits on the causal path between treatment and outcome.  
An **instrumental variable** affects the treatment but has no direct path to the outcome.

In [ ]:
# Variables that mediate the effect of temperature on cavities
print("Mediators (temperature → cavities):", graph.mediators_of("temperature", "cavities"))

# Instrumental variables for the ice cream → cavities relationship
print("IVs (ice cream sales → cavities):", graph.instrumental_variables_for("ice cream sales", "cavities"))

## Expert consensus

Query multiple domain experts **in parallel** — `asyncio.gather` fires all calls simultaneously.  
Edges are merged by vote count, so you can filter for consensus (`min_votes=2` means at least 2 experts agreed).

In [ ]:
# Suggest relevant expert roles (one LLM call)
experts = await suggester.suggest_domain_experts(variables, n_experts=3)
print("Experts:", experts)

In [ ]:
# Build graph with expert consensus — N parallel LLM calls, one per expert
expert_graph = await suggester.suggest_graph(variables, expertise_list=experts)
print(expert_graph)

In [ ]:
# Filter to edges at least 2 experts agreed on
high_confidence = expert_graph.top_edges(min_votes=2)
for (cause, effect), data in high_confidence:
    print(f"{cause} → {effect}  (votes: {data.votes}, avg confidence: {data.avg_confidence:.2f})")

## Confounders

Find variables in your list that **directly cause both** treatment and outcome — 
these are the confounders you need to adjust for.

Set `latent=True` to find **unmeasured** confounders outside your variable list.

In [ ]:
# Observed confounders — searched within `variables`
confounders = await suggester.suggest_confounders(
    treatment="ice cream sales",
    outcome="cavities",
    variables=variables,
)
print("Observed confounders:", confounders)

In [ ]:
# Latent (unmeasured) confounders — LLM suggests factors outside your variable list
latent = await suggester.suggest_confounders(
    treatment="ice cream sales",
    outcome="cavities",
    variables=variables,
    latent=True,
)
print("Latent confounders:", latent)

## Negative controls

Negative controls are variables that *should be unaffected* by the treatment. 
If your model shows a treatment effect on a negative control, something is wrong 
with your causal assumptions — they're a sanity-check.

In [ ]:
negative_controls = await suggester.suggest_negative_controls(
    treatment="ice cream sales",
    outcome="cavities",
    variables=variables,
)
print("Negative controls:", negative_controls)

## Validating the graph

`critique_graph` shows the LLM your proposed edges and asks: *which are genuinely causal, 
and what's missing?* Compare the critique against the original to find confirmed vs disputed edges.

In [ ]:
critique = await suggester.critique_graph(graph, variables, expertise_list=experts)
print("Critique graph:")
print(critique)

In [ ]:
print("Edge-by-edge comparison:")
for (cause, effect) in graph.edges:
    status = "✓ CONFIRMED" if critique.edge_data(cause, effect) else "✗ DISPUTED"
    print(f"  {status}: {cause} → {effect}")

# New edges the critique added
original_edges = set(graph.edges)
new_edges = [(c, e) for (c, e) in critique.edges if (c, e) not in original_edges]
if new_edges:
    print("\nEdges suggested by critique:")
    for (cause, effect) in new_edges:
        print(f"  + {cause} → {effect}")

## Inspecting edge reasoning

Every edge stores the LLM's reasoning from each expert that voted for it. 
Use `reasoning_for` or `edge_data` to inspect *why* an edge was suggested.

In [ ]:
# Full EdgeData: votes, avg confidence, and per-expert reasoning
data = expert_graph.edge_data("temperature", "ice cream sales")
if data:
    print(f"Votes: {data.votes}")
    print(f"Avg confidence: {data.avg_confidence:.2f}")
    print("Reasoning:")
    for r in data.reasonings:
        print(f"  - {r}")

## Switching LLM providers

Because `ModelSuggester` accepts any instructor-wrapped async client, 
changing providers requires **one line** — no other code changes.

In [ ]:
# Anthropic Claude
# from anthropic import AsyncAnthropic
# client = instructor.from_anthropic(AsyncAnthropic())
# suggester = ModelSuggester(client=client, model="claude-opus-4-5", context="cardiovascular health")

# Google Gemini
# import google.generativeai as genai
# client = instructor.from_gemini(genai.GenerativeModel("gemini-1.5-pro"))
# suggester = ModelSuggester(client=client, model="gemini-1.5-pro", context="cardiovascular health")

# Ollama (local, OpenAI-compatible)
# from openai import AsyncOpenAI
# ollama_client = AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# client = instructor.from_openai(ollama_client)
# suggester = ModelSuggester(client=client, model="llama3", context="cardiovascular health")

## Visualizing the graph

`graph.plot()` renders the graph with Graphviz — edge thickness and color encode confidence, and labels show vote counts.

> **Requires Graphviz system binaries:** `brew install graphviz` (macOS) or `sudo apt-get install graphviz` (Linux).

In [ ]:
graph.plot()